In [2]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from trajectory_plotter import TrajectoryValidationRiskPloter

In [1]:
print ("hey")

hey


In [ ]:
# --- Initial run ---
plotter = TrajectoryValidationRiskPloter(max_T=500, random_seed=42)
plotter.X, plotter.y = plotter.generate_data(n=200, p=400)
X_train, y_train, X_valid, y_valid = plotter.split_train_valid()
X_test, y_test = plotter.generate_data(n=1000, p=400)
_, plotter.train_over_time, plotter.valid_over_time, plotter.test_over_time = plotter.run_GD_efficient(
    X_train, y_train, X_valid, y_valid, X_test, y_test, eta=0.1
)
print(f"Ran {len(plotter.train_over_time)} iterations.")

In [ ]:
# --- Build figure ---
ts = list(range(1, len(plotter.train_over_time) + 1))

fig = go.FigureWidget()
fig.add_scatter(x=ts, y=plotter.train_over_time, name='Train Risk',      line=dict(color='steelblue'))
fig.add_scatter(x=ts, y=plotter.valid_over_time, name='Validation Risk', line=dict(color='tomato'))
fig.add_scatter(x=ts, y=plotter.test_over_time,  name='Test Risk',       line=dict(color='seagreen'))
fig.update_layout(
    xaxis_title='Iteration t', yaxis_title='MSE',
    title='GD Trajectory', height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
)

# --- Scale toggles (green = selected) ---
def make_toggle_pair(label):
    b_lin = widgets.ToggleButton(value=True,  description='linear',
                                  button_style='success', layout=widgets.Layout(width='70px', height='28px'))
    b_log = widgets.ToggleButton(value=False, description='log',
                                  button_style='',        layout=widgets.Layout(width='70px', height='28px'))
    def on_lin(change):
        if change['new']:
            b_log.value = False; b_log.button_style = ''; b_lin.button_style = 'success'
        elif not b_log.value:
            b_lin.value = True
        update_scales()
    def on_log(change):
        if change['new']:
            b_lin.value = False; b_lin.button_style = ''; b_log.button_style = 'success'
        elif not b_lin.value:
            b_log.value = True
        update_scales()
    b_lin.observe(on_lin, names='value')
    b_log.observe(on_log, names='value')
    row = widgets.HBox([widgets.Label(label, layout=widgets.Layout(width='50px')), b_lin, b_log])
    return row, b_lin, b_log

x_row, x_lin, x_log = make_toggle_pair('X scale:')
y_row, y_lin, y_log = make_toggle_pair('Y scale:')

def update_scales():
    fig.update_layout(xaxis_type='log' if x_log.value else 'linear',
                      yaxis_type='log' if y_log.value else 'linear')

# --- Parameter inputs ---
def parse_int(s):
    s = str(s).strip().lower()
    if s.endswith('k'):
        return int(float(s[:-1]) * 1000)
    return int(float(s))

st = {'description_width': '42px'}
lo = widgets.Layout(width='118px')
n_input      = widgets.Text(value='200',  description='n:',        style=st, layout=lo)
p_input      = widgets.Text(value='400',  description='p:',        style=st, layout=lo)
seed_input   = widgets.Text(value='42',   description='seed:',     style=st, layout=lo)
eta_input    = widgets.Text(value='0.1',  description='eta:',      style=st, layout=lo)
maxT_input   = widgets.Text(value='500',  description='max_T:',    style=st, layout=lo)
ntest_input  = widgets.Text(value='1000', description='n_test:',   style=st, layout=lo)

run_btn = widgets.Button(description='Rerun', button_style='primary', layout=widgets.Layout(width='75px'))
status  = widgets.Label(value='')

def on_run(b):
    status.value = 'Running...'
    run_btn.disabled = True
    try:
        p = TrajectoryValidationRiskPloter(max_T=parse_int(maxT_input.value),
                                           random_seed=parse_int(seed_input.value))
        p.X, p.y = p.generate_data(n=parse_int(n_input.value), p=parse_int(p_input.value))
        X_tr, y_tr, X_val, y_val = p.split_train_valid()
        X_te, y_te = p.generate_data(n=parse_int(ntest_input.value), p=parse_int(p_input.value))
        _, p.train_over_time, p.valid_over_time, p.test_over_time = p.run_GD_efficient(
            X_tr, y_tr, X_val, y_val, X_te, y_te, eta=float(eta_input.value)
        )
        new_ts = list(range(1, len(p.train_over_time) + 1))
        with fig.batch_update():
            fig.data[0].x = new_ts; fig.data[0].y = p.train_over_time
            fig.data[1].x = new_ts; fig.data[1].y = p.valid_over_time
            fig.data[2].x = new_ts; fig.data[2].y = p.test_over_time
        status.value = f'Done. ({len(p.train_over_time)} iters)'
    except Exception as e:
        status.value = f'Error: {e}'
    run_btn.disabled = False

run_btn.on_click(on_run)

# --- Layout ---
display(widgets.VBox([
    widgets.HBox([x_row, y_row]),
    fig,
    widgets.HTML('<hr style="margin:6px 0">'),
    widgets.HBox([n_input, p_input, seed_input, eta_input, maxT_input, ntest_input]),
    widgets.HBox([run_btn, status]),
]))